# cachepy — Feature Showcase

cachepy is a disk-backed memoization decorator for Python, ported from the R package cacheR.  
It caches function results to disk so expensive computations are only run once.

This notebook walks through all major features.

In [ ]:
import os, sys, time, shutil
from pathlib import Path

# Ensure cachepy is importable
sys.path.insert(0, str(Path.cwd().parent.parent))

from cachepy import cache_file, cache_tree_nodes, cache_tree_reset
from cachepy.cache_file import (
    cache_prune, cache_stats, fast_file_hash,
    _file_state_cache, load_config,
)

# Helper to start fresh
DEMO_CACHE = Path("demo_cache")
def fresh_cache():
    if DEMO_CACHE.exists():
        shutil.rmtree(DEMO_CACHE)
    cache_tree_reset()
    _file_state_cache.clear()
    return DEMO_CACHE

print("cachepy loaded successfully")

## 1. Basic Caching

Wrap any function with `@cache_file()` and its results are cached to disk.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir, backend="pickle")
def slow_computation(n):
    """Simulate an expensive computation."""
    time.sleep(1)
    return sum(i**2 for i in range(n))

# First call — slow (1 second)
t0 = time.time()
result1 = slow_computation(10_000)
print(f"First call:  {result1:,}  ({time.time()-t0:.2f}s)")

# Second call — instant (loaded from disk)
t0 = time.time()
result2 = slow_computation(10_000)
print(f"Second call: {result2:,}  ({time.time()-t0:.4f}s)")

# Different args — cache miss, runs again
t0 = time.time()
result3 = slow_computation(5_000)
print(f"New args:    {result3:,}  ({time.time()-t0:.2f}s)")

## 2. Argument Normalization

Positional and named arguments are normalized — the same logical call always hits the same cache entry.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir, backend="pickle")
def add(a, b, c=0):
    print(f"  -> executing add({a}, {b}, {c})")
    return a + b + c

print("Call 1: add(1, 2)")
add(1, 2)

print("Call 2: add(a=1, b=2)  [same — cache hit]")
add(a=1, b=2)

print("Call 3: add(b=2, a=1)  [reversed names — cache hit]")
add(b=2, a=1)

print("Call 4: add(1, 2, c=0)  [explicit default — cache hit]")
add(1, 2, c=0)

print("Call 5: add(1, 2, c=10)  [different args — cache miss]")
add(1, 2, c=10)

print(f"\nCache entries: {len(list(cache_dir.glob('*.pkl')))}")

## 3. kwargs Order Independence

For `**kwargs` functions, keyword argument order doesn't matter.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir, backend="pickle")
def config_hash(**kwargs):
    print(f"  -> executing with {kwargs}")
    return str(sorted(kwargs.items()))

print("Call 1: config_hash(alpha=0.1, beta=0.9)")
config_hash(alpha=0.1, beta=0.9)

print("Call 2: config_hash(beta=0.9, alpha=0.1)  [reordered — cache hit]")
config_hash(beta=0.9, alpha=0.1)

print(f"\nCache entries: {len(list(cache_dir.glob('*.pkl')))}")

## 4. File Dependency Tracking

When function arguments point to files or directories, cachepy detects content changes automatically.

In [ ]:
cache_dir = fresh_cache()
data_file = Path("demo_data.csv")
data_file.write_text("gene,expr\nTP53,10.5\nBRCA1,8.2\n")

@cache_file(cache_dir, file_args=["fpath"], backend="pickle")
def parse_csv(fpath):
    print(f"  -> parsing {fpath}")
    lines = Path(fpath).read_text().strip().split("\n")
    header = lines[0].split(",")
    rows = [dict(zip(header, l.split(","))) for l in lines[1:]]
    return rows

print("Call 1: parse original file")
result = parse_csv(str(data_file))
print(f"  Result: {result}")

print("\nCall 2: same file, same content — cache hit")
result = parse_csv(str(data_file))

# Modify the file
time.sleep(0.1)
data_file.write_text("gene,expr\nTP53,10.5\nBRCA1,8.2\nEGFR,15.3\n")
_file_state_cache.clear()

print("\nCall 3: file content changed — cache miss")
result = parse_csv(str(data_file))
print(f"  Result: {result}")

data_file.unlink()  # cleanup

## 5. Body Change Detection

If you redefine a function (e.g. fix a bug), the cache is automatically invalidated.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir, backend="pickle")
def transform(x):
    return x * 2  # version 1

print(f"v1: transform(5) = {transform(5)}")

# "Fix" the function — body hash changes
@cache_file(cache_dir, backend="pickle")
def transform(x):
    return x * 3  # version 2

print(f"v2: transform(5) = {transform(5)}")
print(f"Cache entries: {len(list(cache_dir.glob('*.pkl')))}  (2 = both versions)")

## 6. Version Parameter

Manually bump a version string to invalidate cache without changing the function body.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir, backend="pickle", version="1.0")
def model_predict(x):
    print("  -> running model")
    return x * 42

print("Version 1.0:")
model_predict(10)
model_predict(10)  # cache hit

# Bump version (e.g. retrained model, new reference genome, etc.)
@cache_file(cache_dir, backend="pickle", version="2.0")
def model_predict(x):
    print("  -> running model")
    return x * 42

print("\nVersion 2.0:")
model_predict(10)  # cache miss — different version

## 7. `_force` and `_skip_save`

Control caching behavior per-call without changing the decorator.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir, backend="pickle")
def fetch_data(query):
    print(f"  -> fetching '{query}'")
    return {"query": query, "timestamp": time.time()}

print("Normal call:")
r1 = fetch_data("TP53")

print("\nCached call (no execution):")
r2 = fetch_data("TP53")
print(f"  Same timestamp: {r1['timestamp'] == r2['timestamp']}")

print("\nForced re-execution (_force=True):")
r3 = fetch_data("TP53", _force=True)
print(f"  New timestamp: {r3['timestamp'] != r1['timestamp']}")

print("\nDry run — execute but don't save (_skip_save=True):")
before = len(list(cache_dir.glob('*.pkl')))
fetch_data("BRCA1", _skip_save=True)
after = len(list(cache_dir.glob('*.pkl')))
print(f"  Cache files before/after: {before}/{after}  (unchanged)")

## 8. `depends_on_files` and `depends_on_vars`

Declare external dependencies that should invalidate the cache.

In [ ]:
cache_dir = fresh_cache()

# depends_on_files: cache invalidates when a config file changes
config = Path("demo_config.yml")
config.write_text("threshold: 0.05\n")

@cache_file(cache_dir, backend="pickle", depends_on_files=[str(config)])
def analyze(x):
    print("  -> analyzing")
    return x ** 2

print("Call 1:"); analyze(5)
print("Call 2 (cached):"); analyze(5)

config.write_text("threshold: 0.01\n")
_file_state_cache.clear()
print("Call 3 (config changed — re-executes):"); analyze(5)

config.unlink()

# depends_on_vars: cache invalidates when a variable changes
cache_dir2 = fresh_cache()

@cache_file(cache_dir2, backend="pickle", depends_on_vars={"schema": "v3"})
def process(x):
    print("  -> processing")
    return x + 1

print("\nSchema v3:"); process(10)
print("Schema v3 again (cached):"); process(10)

@cache_file(cache_dir2, backend="pickle", depends_on_vars={"schema": "v4"})
def process(x):
    print("  -> processing")
    return x + 1

print("Schema v4 (re-executes):"); process(10)

## 9. Environment Variable Tracking

In [ ]:
cache_dir = fresh_cache()

os.environ["GENOME_BUILD"] = "hg38"

@cache_file(cache_dir, backend="pickle", env_vars=["GENOME_BUILD"])
def align(reads):
    build = os.environ.get("GENOME_BUILD", "unknown")
    print(f"  -> aligning to {build}")
    return f"aligned_{reads}_to_{build}"

print("hg38:"); align("sample1")
print("hg38 again (cached):"); align("sample1")

os.environ["GENOME_BUILD"] = "hg19"
print("hg19 (re-executes):"); align("sample1")

os.environ.pop("GENOME_BUILD", None)

## 10. Verbose Mode

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

cache_dir = fresh_cache()

@cache_file(cache_dir, backend="pickle", verbose=True)
def compute(x):
    return x * 2

compute(1)  # first execution
compute(1)  # cache hit
compute(2)  # cache miss (argument changed)

logging.getLogger().setLevel(logging.WARNING)  # reset

## 11. Cache Statistics and Pruning

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir, backend="pickle")
def big_result(n):
    return list(range(n))

for n in [100, 1000, 10000, 50000]:
    big_result(n)

stats = cache_stats(cache_dir)
print("Cache statistics:")
for k, v in stats.items():
    print(f"  {k}: {v}")

# Pruning: delete entries older than N days
print(f"\nFiles before prune: {len(list(cache_dir.glob('*.pkl')))}")
cache_prune(cache_dir, days_old=0)  # 0 = delete everything
print(f"Files after prune:  {len(list(cache_dir.glob('*.pkl')))}")

## 12. Cache Tree (Dependency Graph)

When cached functions call other cached functions, cachepy tracks the call graph.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir, backend="pickle")
def load_data(path):
    return [1, 2, 3, 4, 5]

@cache_file(cache_dir, backend="pickle")
def normalize(data):
    mean = sum(data) / len(data)
    return [x - mean for x in data]

@cache_file(cache_dir, backend="pickle")
def pipeline(path):
    raw = load_data(path)
    normed = normalize(raw)
    return normed

result = pipeline("input.csv")
print(f"Result: {result}")

nodes = cache_tree_nodes()
print(f"\nGraph nodes: {len(nodes)}")
for nid, node in nodes.items():
    parents = node.get('parents', [])
    children = node.get('children', [])
    print(f"  {node['fname']}  parents={len(parents)}  children={len(children)}")

## 13. Recursive Functions

In [ ]:
cache_dir = fresh_cache()
call_count = 0

@cache_file(cache_dir, backend="pickle")
def fib(n):
    global call_count
    call_count += 1
    if n <= 1:
        return n
    return fib(n-1) + fib(n-2)

call_count = 0
result = fib(10)
print(f"fib(10) = {result}")
print(f"Function calls (first run): {call_count}")

call_count = 0
result = fib(10)
print(f"\nfib(10) = {result}  (from cache)")
print(f"Function calls (second run): {call_count}")

## 14. Error Handling

Functions that raise exceptions don't leave stale cache entries or broken graph nodes.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir, backend="pickle")
def risky(x):
    if x < 0:
        raise ValueError("negative input")
    return x ** 0.5

# Successful call
print(f"risky(4) = {risky(4)}")
print(f"Graph nodes after success: {len(cache_tree_nodes())}")

# Failed call — no stale node left behind
try:
    risky(-1)
except ValueError as e:
    print(f"\nrisky(-1) raised: {e}")
    print(f"Graph nodes after error: {len(cache_tree_nodes())}")
    print(f"Cache files: {len(list(cache_dir.glob('*.pkl')))}  (only the successful one)")

## 15. YAML Configuration

In [ ]:
config_path = Path("demo_cachepy.yml")
config_path.write_text(
    "cache_dir: /tmp/my_project_cache\n"
    "backend: pickle\n"
    "verbose: true\n"
    "env_vars:\n"
    "  - HOME\n"
    "  - GENOME_BUILD\n"
)

config = load_config(config_path)
print("Loaded config:")
for k, v in config.items():
    print(f"  {k}: {v}")

# Existing values take priority
config2 = load_config(config_path, existing={"backend": "qs"})
print(f"\nWith existing backend='pickle': {config2['backend']}")

config_path.unlink()

## Cleanup

## 16. Speed Comparison: First Execution vs Cache Hit

Benchmark showing how caching eliminates redundant computation.

In [ ]:
if DEMO_CACHE.exists():
    shutil.rmtree(DEMO_CACHE)
print("Cleaned up.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cache_dir = fresh_cache()

# Define functions with increasing computation cost
sleep_times = [0.1, 0.25, 0.5, 1.0, 2.0]
first_times = []
cached_times = []

for delay in sleep_times:
    cache_dir_i = fresh_cache()

    @cache_file(cache_dir_i, backend="pickle")
    def work(x, _delay=delay):
        time.sleep(_delay)
        return x ** 2

    # First execution (cold)
    t0 = time.time()
    work(42)
    first_times.append(time.time() - t0)

    # Cached execution (hot)
    t0 = time.time()
    work(42)
    cached_times.append(time.time() - t0)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Bar chart
x = np.arange(len(sleep_times))
width = 0.35
ax = axes[0]
bars1 = ax.bar(x - width/2, [t * 1000 for t in first_times], width, label="First execution", color="#e74c3c")
bars2 = ax.bar(x + width/2, [t * 1000 for t in cached_times], width, label="Cache hit", color="#2ecc71")
ax.set_xlabel("Simulated computation time (s)")
ax.set_ylabel("Wall time (ms)")
ax.set_title("First Execution vs Cache Hit")
ax.set_xticks(x)
ax.set_xticklabels([f"{s}s" for s in sleep_times])
ax.legend()
ax.set_yscale("log")
ax.grid(axis="y", alpha=0.3)

# Speedup factor
speedups = [f / c for f, c in zip(first_times, cached_times)]
ax2 = axes[1]
ax2.bar(x, speedups, color="#3498db", width=0.5)
ax2.set_xlabel("Simulated computation time (s)")
ax2.set_ylabel("Speedup factor (x)")
ax2.set_title("Cache Speedup")
ax2.set_xticks(x)
ax2.set_xticklabels([f"{s}s" for s in sleep_times])
ax2.grid(axis="y", alpha=0.3)
for i, s in enumerate(speedups):
    ax2.text(i, s + max(speedups) * 0.02, f"{s:.0f}x", ha="center", fontsize=10, fontweight="bold")

fig.tight_layout()
plt.show()

print(f"\nCache hit overhead: {np.mean(cached_times)*1000:.1f} ms avg")